# ROUND 4 — bot-name alpha discovery

Round 4 exposes counterparty bot identities (`Mark 01` … `Mark 67`) on every public-tape print AND on our own fills.  This notebook profiles each bot, mines lead-lag, and proposes the resulting trader strategy.

Companion files in this folder:
* `bot_alpha_analysis.py` — the same analysis as a CLI script (`uv run python notebooks/round4/bot_alpha_analysis.py`).
* `round4_strats.txt` — final write-up: strategy book, untapped alphas, probe-trader plan.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('..').resolve().parent
DATA = ROOT / 'data' / 'ROUND_4'
DAYS = (1, 2, 3)

trades = pd.concat([
    pd.read_csv(DATA / f'trades_round_4_day_{d}.csv', sep=';').assign(day=d)
    for d in DAYS
], ignore_index=True)
prices = pd.concat([
    pd.read_csv(DATA / f'prices_round_4_day_{d}.csv', sep=';').assign(day=d)
    for d in DAYS
], ignore_index=True)

trades['day'].nunique(), len(trades), trades['symbol'].nunique()

## 1. Net flow per bot per product

Pure-directional bots are the ones with massively unbalanced buys/sells on a single product.

In [ ]:
bots = sorted(set(trades['buyer']) | set(trades['seller']))
prods = sorted(trades['symbol'].unique())
rows = []
for b in bots:
    row = {'bot': b}
    for p in prods:
        bq = trades.loc[(trades['symbol']==p)&(trades['buyer']==b), 'quantity'].sum()
        sq = trades.loc[(trades['symbol']==p)&(trades['seller']==b), 'quantity'].sum()
        row[p] = int(bq - sq)
    rows.append(row)
net_flow = pd.DataFrame(rows).set_index('bot')
net_flow

Reads:
- **Mark 67**: pure VELVET buyer (+1510 net, never sold) → take-the-offer informed flow.
- **Mark 49**: pure VELVET seller (-956 net) → mirror of Mark 67.
- **Mark 22**: BASKET LIQUIDATOR — heavy net sells across every voucher strike + VELVET (-551).
- **Mark 01**: PURE ACCUMULATOR — heavy net buyer of out-of-the-money vouchers.  Almost 1-for-1 with Mark 22's dumps.
- **Mark 14 / 38 / 55**: tight 2-sided MMs (close to net 0 on volume).

## 2. Lead-lag: does a bot's print predict the next 100-tick mid move?

In [ ]:
def lead_lag(product, horizons=(100, 500, 1000, 5000)):
    pp = (prices[prices['product']==product]
          [['day','timestamp','mid_price']].sort_values(['day','timestamp']).reset_index(drop=True))
    tp = trades[trades['symbol']==product]
    out = []
    for b in sorted(set(tp['buyer'])|set(tp['seller'])):
        for side in ('buy','sell'):
            evs = tp[tp['buyer' if side=='buy' else 'seller']==b]
            if len(evs)==0:
                continue
            row = {'bot': b, 'side': side, 'n': len(evs)}
            for h in horizons:
                moves = []
                for _, r in evs.iterrows():
                    fut = pp[(pp['day']==r['day'])&(pp['timestamp']>=r['timestamp']+h)]
                    if len(fut):
                        d = fut.iloc[0]['mid_price'] - r['price']
                        moves.append(d if side=='buy' else -d)
                if moves:
                    row[f'mean_h{h}'] = np.mean(moves)
                    row[f'hit_h{h}'] = float(np.mean([m>0 for m in moves]))
            out.append(row)
    return pd.DataFrame(out)

lead_lag('VELVETFRUIT_EXTRACT').round(3)

In [ ]:
lead_lag('HYDROGEL_PACK').round(3)

Top signals (by hit rate × magnitude):

| product | bot | side | mean h=100 | hit |
|---|---|---|---|---|
| VELVET | Mark 67 | buy | +1.17 | 85% |
| VELVET | Mark 49 | sell | -1.23 | 90% |
| VELVET | Mark 22 | sell | -0.80 | 82% |
| HYDRO | Mark 14 | any | +8.0 | 100% |
| HYDRO | Mark 38 | any | -8.0 | 99% |

The HYDRO bot prints are almost-deterministic — Mark 14/38's tight MM lives at the wall and gets blown through 100% of the time within 100 ticks.

## 3. Mark 22 basket dumps

A burst is defined as Mark 22 selling >=4 different products at the same timestamp.  These are the moments to be a buyer.

In [ ]:
sells22 = trades[trades['seller']=='Mark 22']
g = sells22.groupby(['day','timestamp'])['symbol'].agg(list)
bursts = [(d,t,sorted(s)) for (d,t),s in g.items() if len(s)>=4]
f'Total Mark 22 basket bursts (d1+d2+d3, >=4 legs): {len(bursts)}'

## 4. own_trades vs market_trades — disjoint

From `prosperity4bt/runner.py @ match_orders`:

```python
market_trades = { ... }                    # raw tape this tick
for product in data.products:
    new_trades = []
    for order in orders.get(product, []):
        new_trades.extend(match_order(order, market_trades.get(product, []), ...))
    if new_trades:
        state.own_trades[product] = new_trades       # what we filled
for product, trades in market_trades.items():
    for t in trades:
        t.trade.quantity = min(t.buy_quantity, t.sell_quantity)
    state.market_trades[product] = [t.trade for t in trades if t.trade.quantity > 0]
```

So matched volume is **moved** from `market_trades` into `own_trades`.  They never overlap.

Implication for ROUND_4 bot signals:
* On the local backtester, scoring against `market_trades` alone undercounts informed prints we participated in.
* On the live IMC site, both channels carry counterparty names (the open-source backtester replicates this when run with bot names — verify via `trader_PROBE.py`).
* For VELVET we found scoring against `market_trades` only is slightly cleaner (own-trades fades real entries).  For HYDRO the union helps.  HYPER uses each accordingly.

## 5. Final per-product trader breakdown (from `uv run rank --round 4 --show-per-product`)

After all this analysis the resulting `trader_HYPER.py` ranks #1:

```
rank  trader                 total     avg/day   d1        d2        d3
1     trader_HYPER.py        785,117   261,706   341,589   243,421   200,108
2     trader_BOTVELVET.py    779,833   259,944   341,891   237,935   200,008
3     trader_WAITALPHA.py    778,200   259,400   342,142   236,179   199,880
4     trader_BUGALPHA.py     777,631   259,210   342,142   233,800   201,690
```

HYPER's gain over BOTVELVET is concentrated on HYDROGEL_PACK (`+$4,475`) thanks to a 12-tick additive bot tilt that flips the swing-edge entry near threshold 14.  See `round4_strats.txt`.